# Letters to TEI XML with an LLM on the DHinfra cluster

Ten transcribed letters (German, French, English, Italian) from the Hammer-Purgstall
correspondence (1794-1856) go in, TEI XML for the letter structure (opener, dateline,
salutations, paragraphs, closer, signature, postscript, address) comes out.

The model is `deepseek-v4.1-flash`, called through the cluster's OpenAI-compatible API. The
prompt (task, encoding rules, five input-output examples) is taken from the paper
linked in the README. No GPU is needed in your own
session: the model runs on the cluster.

**Steps:** 1. set up, 2. look at the prompt, 3. encode one letter, 4. encode all ten,
5. check the results.

## 1. Set up

In [ ]:
# Run once per session. The cluster's Python environment may already have these.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

Your project token is read from `DHINFRA_KEY`, either from the environment or from a `.env`
file next to this notebook (`DHINFRA_KEY="..."`). Never commit the `.env` file.

In [ ]:
from pathlib import Path
import concurrent.futures as cf

import pandas as pd

import helpers

MODEL = "deepseek-v4.1-flash"   # pick a name from the list printed below
THINKING = False             # True: the model reasons first (slower, more tokens)
WORKERS = 4                  # parallel requests; keep this small, the cluster is shared

PROMPT_DIR = Path("prompts/letter_structure")
LETTER_DIR = Path("data/letters")
OUTPUT_DIR = Path("output") / MODEL
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = helpers.get_client()
# Optional: list the models the API offers, to check that MODEL is spelled correctly.
print([m.id for m in client.models.list().data])

## 2. The prompt

Three files make up the *system prompt*: `prompt.txt` (the task), `encoding_rules.txt`
and `few_shot_examples.txt` (five letters with their TEI encoding). The *user message* is a
one-line instruction from `user_message.txt` followed by the letter.

In [ ]:
system_prompt = helpers.build_system_prompt(PROMPT_DIR)
print(f"{len(system_prompt):,} characters")
print(system_prompt[:1500], "...")

## 3. One letter

`letter1.txt` is a short German petition. The result should be a `<body>` element with a
`<div type="letter">` inside.

In [ ]:
letter_path = LETTER_DIR / "letter1.txt"
letter_text = letter_path.read_text(encoding="utf-8")
print(letter_text)

In [ ]:
xml_text, usage = helpers.ask_model(
    client, MODEL, system_prompt,
    helpers.build_user_message(PROMPT_DIR, letter_text),
    thinking=THINKING,
)
print(xml_text)
print()
print("tokens in/out:", usage.prompt_tokens, usage.completion_tokens)

## 4. All ten letters

The letters are sent as separate requests, a few at a time. Each result is written to
`output/<model>/<letter>.xml`. The system prompt is the same for every request, which is
why it is built once above.

In [ ]:
def run_one(path):
    text = path.read_text(encoding="utf-8")
    xml, usage = helpers.ask_model(
        client, MODEL, system_prompt, helpers.build_user_message(PROMPT_DIR, text),
        thinking=THINKING,
    )
    (OUTPUT_DIR / f"{path.stem}.xml").write_text(xml, encoding="utf-8")
    return path.stem, text, xml, usage.prompt_tokens, usage.completion_tokens

letters = sorted(LETTER_DIR.glob("*.txt"))
results = {}
with cf.ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futures = {pool.submit(run_one, p): p for p in letters}
    for future in cf.as_completed(futures):
        name = futures[future].stem
        try:
            results[name] = future.result()
            print("done  ", name)
        except Exception as err:   # keep going; one failed letter should not stop the batch
            print("FAILED", name, "-", err)

## 5. Check the results

There is no reference encoding in this example, so the checks are cheap ones:

- **well_formed**: the output parses as XML.
- **text_kept_%**: how much of the letter's wording is still in the output, in order. The
  task is to encode, not to shorten or correct, so this should be close to 100.
- **elements**: which structural elements were used and how often.

Whether the encoding is *right* needs a look at the XML, see below.

In [ ]:
rows = []
for name, (_, text, xml, tokens_in, tokens_out) in sorted(results.items()):
    rows.append({"letter": name, **helpers.check_output(xml, text),
                 "tokens_in": tokens_in, "tokens_out": tokens_out})
pd.DataFrame(rows).set_index("letter")

In [ ]:
# Look at one result next to its source. Change the name to see another letter.
name = "letter499"
print(results[name][1])
print("-" * 60)
print(results[name][2])

## Where next

- Change `MODEL`, or the files in `prompts/`, and run again. The output goes to a folder named
  after the model, so runs do not overwrite each other.
- Put your own plain-text letters into `data/letters/`.
- For a long batch, run the same code as a script in a SLURM job instead of in the notebook.